<a href="https://colab.research.google.com/github/xXLukaENPXx/Algoritmos-2/blob/main/KNN_con_librer%C3%ADas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Cargar los datos directamente desde la URL
url = "https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv"
df = pd.read_csv(url)

# Mostrar información básica del dataset
print("=== INFORMACIÓN DEL DATASET ===")
print(f"Dimensiones del dataset: {df.shape}")
print(f"Columnas disponibles: {df.columns.tolist()}")
print(f"Participantes únicos: {df['Participante'].unique()}")
print(f"Total de participantes: {df['Participante'].nunique()}")
print(f"Total de cortes (filas): {len(df)}")
print(f"Cortes por participante (idealmente 50): {df.groupby('Participante').size()}")
print("\n" + "="*50 + "\n")

# --- PREPARACIÓN DE DATOS ---
# Identificar columnas a excluir
columnas_a_excluir = ['Minuto', 'EMG', 'Participante']
# La columna 'Grupo' es la etiqueta/clase (0=sano, 1=enfermo)

# Obtener todas las columnas de características (todas excepto las excluidas y la etiqueta)
columnas_caracteristicas = [col for col in df.columns if col not in columnas_a_excluir + ['Grupo']]

print("=== COLUMNAS UTILIZADAS PARA EL ANÁLISIS ===")
print(f"Características ({len(columnas_caracteristicas)} columnas):")
print(columnas_caracteristicas)
print(f"\nVariable objetivo: 'Grupo' (0 = sano, 1 = enfermo)")
print(f"Columnas excluidas: {columnas_a_excluir}")
print("\n" + "="*50 + "\n")

# Verificar si hay valores nulos
print("=== VERIFICACIÓN DE VALORES NULOS ===")
print(df[columnas_caracteristicas + ['Grupo']].isnull().sum().sum())
if df[columnas_caracteristicas + ['Grupo']].isnull().sum().sum() > 0:
    df = df.dropna(subset=columnas_caracteristicas + ['Grupo'])
    print("Se eliminaron filas con valores nulos")
else:
    print("No hay valores nulos en las columnas seleccionadas")
print("\n" + "="*50 + "\n")

# Obtener lista única de participantes
participantes = df['Participante'].unique()
n_participantes = len(participantes)

# Diccionario para almacenar resultados por cada valor de k
resultados_por_k = {}

# Valores de k a evaluar
valores_k = [1, 3, 5]

for k in valores_k:
    print(f"\n=== EJECUTANDO VALIDACIÓN PARA K={k} ===")

    # Listas para almacenar resultados
    accuracy_por_paciente = []
    clasificaciones_por_paciente = {}  # Diccionario: paciente -> lista de 0/1 por corte

    # Loop sobre cada paciente (dejar uno fuera)
    for i, paciente_test in enumerate(participantes):
        # Separar datos de prueba (paciente actual) y entrenamiento (todos los demás)
        test_mask = df['Participante'] == paciente_test
        train_mask = ~test_mask

        X_train = df.loc[train_mask, columnas_caracteristicas].values
        y_train = df.loc[train_mask, 'Grupo'].values
        X_test = df.loc[test_mask, columnas_caracteristicas].values
        y_test = df.loc[test_mask, 'Grupo'].values

        # Verificar que el paciente tenga exactamente 50 cortes
        n_cortes_test = len(y_test)
        if n_cortes_test != 50:
            print(f"  Nota: Paciente {paciente_test} tiene {n_cortes_test} cortes (se esperaban 50)")

        # Normalizar las características (importante para KNN)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Crear y entrenar el modelo KNN
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train_scaled, y_train)

        # Predecir
        y_pred = knn.predict(X_test_scaled)

        # Calcular métricas
        accuracy = accuracy_score(y_test, y_pred)
        accuracy_por_paciente.append(accuracy)

        # Guardar clasificaciones individuales (1 = correcto, 0 = incorrecto)
        clasificaciones_ind = (y_pred == y_test).astype(int).tolist()
        clasificaciones_por_paciente[paciente_test] = clasificaciones_ind

        # Mostrar progreso
        if (i+1) % 5 == 0 or i == n_participantes-1:
            print(f"  Procesados {i+1}/{n_participantes} pacientes")

    # Guardar resultados para este k
    resultados_por_k[k] = {
        'accuracies': accuracy_por_paciente,
        'clasificaciones': clasificaciones_por_paciente,
        'media': np.mean(accuracy_por_paciente),
        'std': np.std(accuracy_por_paciente)
    }

    print(f"\n  Resultados para K={k}:")
    print(f"  Media de precisión: {resultados_por_k[k]['media']:.4f}")
    print(f"  Desviación estándar: {resultados_por_k[k]['std']:.4f}")

# --- MOSTRAR RESULTADOS DETALLADOS ---
print("\n" + "="*60)
print("=== RESULTADOS FINALES POR PACIENTE ===")
print("="*60)

for k in valores_k:
    print(f"\n--- K={k} ---")
    print(f"Precisión media global: {resultados_por_k[k]['media']:.4f} ± {resultados_por_k[k]['std']:.4f}")
    print("\nPrecisiones individuales por paciente:")

    for paciente, acc in zip(participantes, resultados_por_k[k]['accuracies']):
        print(f"  {paciente}: {acc:.4f} ({int(acc*50)}/50 cortes correctos)")

# --- CREAR TABLA COMPARATIVA ---
print("\n" + "="*60)
print("=== TABLA COMPARATIVA K=1, K=3, K=5 ===")
print("="*60)

tabla_comparativa = pd.DataFrame({
    'Paciente': participantes
})

for k in valores_k:
    tabla_comparativa[f'Acc_K{k}'] = resultados_por_k[k]['accuracies']

# Calcular estadísticas adicionales
stats = {
    'Paciente': ['MEDIA', 'STD', 'MÍNIMO', 'MÁXIMO']
}
for k in valores_k:
    accs = resultados_por_k[k]['accuracies']
    stats[f'Acc_K{k}'] = [
        np.mean(accs),
        np.std(accs),
        np.min(accs),
        np.max(accs)
    ]

tabla_comparativa = pd.concat([tabla_comparativa, pd.DataFrame(stats)], ignore_index=True)

# Mostrar tabla con formato
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("\n" + tabla_comparativa.to_string(index=False))

# --- MOSTRAR EJEMPLO DE CLASIFICACIONES INDIVIDUALES (primeros 5 pacientes) ---
print("\n" + "="*60)
print("=== EJEMPLO DE CLASIFICACIONES CORTE POR CORTE (K=3) ===")
print("(1 = correcto, 0 = incorrecto)")
print("="*60)

ejemplo_pacientes = list(participantes)[:5]  # Primeros 5 pacientes
for paciente in ejemplo_pacientes:
    clasif = resultados_por_k[3]['clasificaciones'][paciente]
    print(f"\n{paciente}: {clasif[:20]}... (primeros 20 de {len(clasif)} cortes)")
    print(f"  Correctos: {sum(clasif)}/{len(clasif)} = {sum(clasif)/len(clasif):.4f}")

# --- ANÁLISIS BREVE ---
print("\n" + "="*60)
print("=== ANÁLISIS DE RESULTADOS ===")
print("="*60)

mejor_k = max(valores_k, key=lambda k: resultados_por_k[k]['media'])
print(f"\nMejor valor de K: {mejor_k} (precisión media: {resultados_por_k[mejor_k]['media']:.4f})")

for k in valores_k:
    media = resultados_por_k[k]['media']
    std = resultados_por_k[k]['std']
    print(f"\nK={k}: {media:.4f} ± {std:.4f}")

print("\n" + "="*60)

=== INFORMACIÓN DEL DATASET ===
Dimensiones del dataset: (900, 25)
Columnas disponibles: ['Grupo', 'Participante', 'Minuto', 'C3', 'C4', 'CZ', 'EMG', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']
Participantes únicos: ['EMV' 'GH2' 'GUR' 'JAL' 'JAN' 'MGN' 'MJN' 'MMA' 'RAN' 'VCN' 'AEF' 'CLM'
 'FGV' 'JGM' 'LIV' 'PCM' 'RLM' 'RRM']
Total de participantes: 18
Total de cortes (filas): 900
Cortes por participante (idealmente 50): Participante
AEF    50
CLM    50
EMV    50
FGV    50
GH2    50
GUR    50
JAL    50
JAN    50
JGM    50
LIV    50
MGN    50
MJN    50
MMA    50
PCM    50
RAN    50
RLM    50
RRM    50
VCN    50
dtype: int64


=== COLUMNAS UTILIZADAS PARA EL ANÁLISIS ===
Características (21 columnas):
['C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']

Variable objetivo: 'Grupo' (0 = sano, 1 = enfermo)
Columnas excluidas: ['Minuto', 'EMG', 'Pa

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# ==============================
# 1. Cargar dataset
# ==============================

url = "https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv"
df = pd.read_csv(url)

# ==============================
# 2. Limpieza de columnas
# ==============================

# Columnas a excluir
# Corrected column names to match the DataFrame
exclude_cols = ["Minuto", "EMG", "Participante"]

# eliminar si existen
exclude_cols = [c for c in exclude_cols if c in df.columns]

# columna objetivo
# Corrected to 'Grupo' as specified in cell qwtOB2BrYLJP
target_col = 'Grupo'

# Features iniciales
feature_cols = [c for c in df.columns if c not in exclude_cols + [target_col]]

# eliminar columnas con NaN
# Note: The previous cell's output indicated no nulls in relevant columns, so this line might be redundant.
df = df.dropna(axis=1)

# Re-filter feature_cols after dropping columns
feature_cols = [c for c in feature_cols if c in df.columns]

print("Columnas utilizadas como características:")
print(feature_cols)

# ==============================
# 3. Identificar participantes
# ==============================

# Corrected column name 'participant' to 'Participante'
participants = df["Participante"].unique()

# guardar resultados
results = {1: {}, 3: {}, 5: {}}

# ==============================
# 4. Leave-One-Patient-Out
# ==============================

for patient in participants:

    # Test set
    # Corrected column name 'participant' to 'Participante'
    test_df = df[df["Participante"] == patient]

    # Train set
    # Corrected column name 'participant' to 'Participante'
    train_df = df[df["Participante"] != patient]

    X_train = train_df[feature_cols].values
    y_train = train_df[target_col].values

    X_test = test_df[feature_cols].values
    y_test = test_df[target_col].values

    # Escalado
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    for k in [1,3,5]:

        knn = KNeighborsClassifier(n_neighbors=k)

        # entrenamiento
        knn.fit(X_train, y_train)

        # predicción
        y_pred = knn.predict(X_test)

        # vector correcto / incorrecto
        correct_vector = (y_pred == y_test).astype(int)

        # accuracy paciente
        acc = correct_vector.sum() / len(correct_vector)

        results[k][patient] = {
            "vector": correct_vector.tolist(),
            "accuracy": acc
        }

# ==============================
# 5. Mostrar resultados
# ==============================

for k in [1,3,5]:

    print("\n========================")
    print(f"K = {k}")
    print("========================")

    accuracies = []

    for patient in results[k]:

        vec = results[k][patient]["vector"]
        acc = results[k][patient]["accuracy"]

        accuracies.append(acc)

        print(f"\nPaciente: {patient}")
        print("Vector:", vec)
        print("Accuracy:", round(acc,3))

    print("\nAccuracy promedio:", round(np.mean(accuracies),3))

Columnas utilizadas como características:
['C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']

K = 1

Paciente: EMV
Vector: [1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1]
Accuracy: 0.88

Paciente: GH2
Vector: [1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0]
Accuracy: 0.24

Paciente: GUR
Vector: [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1]
Accuracy: 0.48

Paciente: JAL
Vector: [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Accuracy: 0.04

Paciente: JAN
Vector: [1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0